In [ ]:
# Run from the repository root, or open this notebook anywhere inside it.
from pathlib import Path
import os
_repo_candidates = [Path.cwd(), *Path.cwd().parents]
_project_root = next((p for p in _repo_candidates if (p / "code/data_process").is_dir() and (p / "README.md").is_file()), None)
if _project_root is None:
    raise RuntimeError("Open this notebook from inside the repository.")
os.chdir(_project_root)


In [ ]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
pd.set_option('display.max_columns', None)


In [ ]:
# Define the base path
base_path = "./data-all-waves"
file_path = os.path.join(base_path, "df_final.csv")
df_final = pd.read_csv(file_path)

In [ ]:
# 2. Merge with UR data

import os
import pandas as pd

# Define the path and read the CSV file
base_path = "./data-all-waves"
ur_ba_path = os.path.join(base_path, "ur_ba_16y.csv")

# Read the CSV file
ur_ba_16y = pd.read_csv(ur_ba_path)

# Rename columns
ur_ba_16y = ur_ba_16y.rename(columns={'CGBD16O': 'UR'})

# First prepare yearly unemployment rates for both degree types
# For Bachelor's
ur_ba_16y['observation_date'] = pd.to_datetime(ur_ba_16y['observation_date'])
ur_ba_16y['year'] = ur_ba_16y['observation_date'].dt.year


# Assume there is a 'YEAR' column in ur_ba_16y indicating the year
# If not, check if 'BAYR' corresponds to a year column
if 'year' in ur_ba_16y.columns:
    # Compute the annual average unemployment rate
    ur_avg = ur_ba_16y.groupby('year')['UR'].mean().reset_index()
    
    # Rename YEAR to BAYR so it matches df_final
    ur_avg = ur_avg.rename(columns={'year': 'BAYR'})
else:
    print("Error: No 'year' column found in ur_ba_16y. Please check the dataset structure.")

# Merge with df_final based on BAYR
df_final= df_final.merge(ur_avg, on='BAYR', how='left')

In [ ]:
# Function to get valid salary range for each year
def get_valid_salary_range(data, year):
   year_data = data[data['REFYR'] == year]
   p1 = year_data['SALARY'].quantile(0.01)
   p99 = year_data['SALARY'].quantile(0.99)
   return p1, p99

# # Print sample sizes before trimming
# print("Before trimming:")
# print(df_final.groupby('REFYR').size())

# Remove outliers by year
clean_salary_dfs = []
for year in df_final['REFYR'].unique():
   year_data = df_final[df_final['REFYR'] == year].copy()
   p1, p99 = get_valid_salary_range(df_final, year)
#    print(f"\nYear {year}:")
#    print(f"1st percentile: ${p1:,.2f}")
#    print(f"99th percentile: ${p99:,.2f}")
   clean_year_data = year_data[(year_data['SALARY'] >= p1) & (year_data['SALARY'] <= p99)]
   clean_salary_dfs.append(clean_year_data)

# Combine cleaned data
df_clean_salary = pd.concat(clean_salary_dfs)

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf
# from linearmodels import IV2SLS

def prepare_data(df, exp_range):
    """
    更灵活的数据预处理
    """
    # 1. Create log salary
    df['log_salary'] = np.log(df['SALARY'])

    # 2. Create experience variables
    df['exp'] = df['REFYR'] - df['BAYR']
    df['exp2'] = df['exp']**2

    # 3. Create dummy for master's degree
    df['master'] = (df['DGRDG'] == 2).astype(int)
    
    # 4. restrict to exp [?,?]
    df = df[df['exp'].between(*exp_range)].copy()
    df['immediate_masters'] = ((df['DGRDG'] == 2) & 
                             (df['DGRYR'] - df['BAYR'] <= 3)).astype(int)
    
    # 自动生成交互项
    df['gradur_master'] = df['gradur'] * df['master']
    df['gradur_exp'] = df['gradur'] * df['exp']
    df['master_exp'] = df['master'] * df['exp']
    df['gradur_master_exp'] = df['gradur'] * df['master'] * df['exp']

    # 1. 中心化连续变量
    df['exp_c'] = df['exp'] - df['exp'].mean()
    df['gradur_c'] = df['gradur'] - df['gradur'].mean()
    df['UR_c'] = df['UR'] - df['UR'].mean()

    # 2. 构建交互项
    df['gradur_master_c'] = df['gradur_c'] * df['master'] 
    df['gradur_exp_c'] = df['gradur_c'] * df['exp_c']
    df['master_exp_c'] = df['master'] * df['exp_c']
    df['gradur_master_exp_c'] = df['gradur_c'] * df['master'] * df['exp_c']
    
    
    # Create year dummies
    year_dummies = pd.get_dummies(df['REFYR'], prefix='year')
    base_year = min(df['REFYR'])
    year_dummies = year_dummies.drop(f'year_{base_year}', axis=1)
    
    return df

def run_ols_analysis(df, formula):
    """
    支持任意公式的OLS回归
    """
    model = smf.ols(formula, data=df)
    results = model.fit(cov_type='HC1')
    return results
def run_ols_robust_analysis(df, formula):
    """
    支持任意公式的OLS回归
    """
    model = smf.ols(formula, data=df)
    results = model.fit(cov_type='cluster', cov_kwds={'groups': df['assigned_id']})
    return results

In [ ]:
# 数据准备
df_processed = prepare_data(df_clean_salary, (3, 14))

In [ ]:
def prepare_data_oreo_flexible_groups(df, exp_groups):
    """
    Prepare data with experience years grouped into flexible bins
    
    Parameters:
    -----------
    df : pandas DataFrame
        The input data
    exp_groups : list of tuples
        List of (start, end) tuples defining experience groups, e.g. [(3,5), (6,8), (9,11), (12,14)]
    """
    # Filter to keep only relevant experience years
    min_exp = min(start for start, _ in exp_groups)
    max_exp = max(end for _, end in exp_groups)
    df = df[df['exp'].between(min_exp, max_exp)].copy()
    
    # Create experience bin variable (maps each experience year to its group)
    df['exp_bin'] = None
    for start, end in exp_groups:
        bin_value = f"{start}_{end}"
        df.loc[(df['exp'] >= start) & (df['exp'] <= end), 'exp_bin'] = bin_value
    
    # Create interaction terms for each bin
    for start, end in exp_groups:
        bin_name = f"{start}_{end}"
        
        # Create binary indicator for this bin
        df[f'in_exp_{bin_name}'] = (df['exp'] >= start) & (df['exp'] <= end)
        
        # Create interaction variables
        df[f'ur_exp_{bin_name}'] = df['UR'] * df[f'in_exp_{bin_name}']
        df[f'master_exp_{bin_name}'] = df['master'] * df[f'in_exp_{bin_name}']
        df[f'master_ur_exp_{bin_name}'] = df['UR'] * df['master'] * df[f'in_exp_{bin_name}']
    
    # Create year dummies
    year_dummies = pd.get_dummies(df['REFYR'], prefix='year')
    base_year = min(df['REFYR'])
    year_dummies = year_dummies.drop(f'year_{base_year}', axis=1)
    df = pd.concat([df, year_dummies], axis=1)
    
    return df

def create_oreo_formula_flexible_groups(exp_groups):
    """
    Create formula with flexible experience groups
    
    Parameters:
    -----------
    exp_groups : list of tuples
        List of (start, end) tuples defining experience groups, e.g. [(3,5), (6,8), (9,11), (12,14)]
    """
    # Base components
    formula = 'log_salary ~ C(REFYR) + C(exp_bin)'
    
    # Add grouped terms
    for start, end in exp_groups:
        bin_name = f"{start}_{end}"
        
        # Add main unemployment rate effect for this bin
        formula += f' + ur_exp_{bin_name}'
        
        # Add master's effect for this bin
        formula += f' + master_exp_{bin_name}'
        
        # Add interaction effect for this bin
        formula += f' + master_ur_exp_{bin_name}'
    
    return formula

def plot_marginal_effects_flexible_groups(results, exp_groups):
    """
    Plot combined effects showing recession impact for BA vs MA with flexible experience groups
    
    Parameters:
    -----------
    results : statsmodels results object
    exp_groups : list of tuples
        List of (start, end) tuples defining experience groups, e.g. [(3,5), (6,8), (9,11), (12,14)]
    """
    coef = results.params
    se = results.bse
    
    # Calculate bin centers for plotting
    bin_centers = [(start + end) / 2 for start, end in exp_groups]
    
    # Calculate marginal effects
    ba_effects = []
    ma_effects = []
    ba_se = []
    ma_se = []
    
    for start, end in exp_groups:
        bin_name = f"{start}_{end}"
        
        # Get coefficients for this bin
        ur_coef = coef.get(f'ur_exp_{bin_name}', 0)
        ur_ma_coef = coef.get(f'master_ur_exp_{bin_name}', 0)
        
        # Get standard errors
        ur_se = se.get(f'ur_exp_{bin_name}', 0)
        ur_ma_se = se.get(f'master_ur_exp_{bin_name}', 0)
        
        # Calculate effects
        ba_effects.append(ur_coef)
        ma_effects.append(ur_coef + ur_ma_coef)
        
        # Calculate standard errors (simplified)
        ba_se.append(ur_se)
        ma_se.append(np.sqrt(ur_se**2 + ur_ma_se**2))
    
    plt.figure(figsize=(12,7))
    
    # Plot BA effects
    plt.plot(bin_centers, ba_effects, 'b-o', label="Bachelor's")
    plt.fill_between(bin_centers, 
                     [eff - 1.96*s for eff, s in zip(ba_effects, ba_se)],
                     [eff + 1.96*s for eff, s in zip(ba_effects, ba_se)],
                     alpha=0.2, color='blue')
    
    # Plot MA effects
    plt.plot(bin_centers, ma_effects, 'r-o', label="Master's")
    plt.fill_between(bin_centers, 
                     [eff - 1.96*s for eff, s in zip(ma_effects, ma_se)],
                     [eff + 1.96*s for eff, s in zip(ma_effects, ma_se)],
                     alpha=0.2, color='red')
    
    # Add bin range labels to x-axis
    plt.xticks(bin_centers, [f"{start}-{end}" for start, end in exp_groups])
    
    # Add reference line at zero
    plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    
    # Add labels and legend
    plt.title("Marginal Effects of Initial Unemployment Rate by Education and Potential Experience")
    plt.xlabel("Years since Graduation from College (Grouped)")
    plt.ylabel("Effect on Log Earnings")
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    return plt

def display_grouped_results_flexible(results, exp_groups):
    """
    Display regression results organized by coefficient type
    
    Parameters:
    -----------
    results : statsmodels results object
    exp_groups : list of tuples
        List of (start, end) tuples defining experience groups, e.g. [(3,5), (6,8), (9,11), (12,14)]
    """
    coef = results.params
    se = results.bse
    pvalues = results.pvalues
    
    # Initialize containers for different coefficient types
    ur_coefs = []
    master_coefs = []
    master_ur_coefs = []
    
    # Get coefficients by type
    for start, end in exp_groups:
        bin_name = f"{start}_{end}"
        
        # Unemployment rate effects
        ur_var = f'ur_exp_{bin_name}'
        if ur_var in coef:
            ur_coefs.append((
                ur_var, 
                coef[ur_var], 
                se[ur_var], 
                pvalues[ur_var]
            ))
        
        # Master's main effects
        master_var = f'master_exp_{bin_name}'
        if master_var in coef:
            master_coefs.append((
                master_var, 
                coef[master_var], 
                se[master_var], 
                pvalues[master_var]
            ))
        
        # Master's interaction with unemployment
        master_ur_var = f'master_ur_exp_{bin_name}'
        if master_ur_var in coef:
            master_ur_coefs.append((
                master_ur_var, 
                coef[master_ur_var], 
                se[master_ur_var], 
                pvalues[master_ur_var]
            ))
    
    # Print results in organized format
    print("\n" + "="*80)
    print("REGRESSION RESULTS BY COEFFICIENT TYPE")
    print("="*80)
    
    # Function to print coefficient group
    def print_coef_group(title, coef_list):
        print(f"\n{title}")
        print("-"*80)
        print(f"{'Variable':<25} {'Coefficient':>12} {'Std Error':>12} {'P-value':>10} {'Significance':>12}")
        print("-"*80)
        
        for var, coef, se, pval in coef_list:
            stars = ""
            if pval < 0.01:
                stars = "***"
            elif pval < 0.05:
                stars = "**"
            elif pval < 0.1:
                stars = "*"
                
            # Extract the experience range from variable name for cleaner display
            var_clean = var.split('_')[-1]
            
            print(f"Experience {var_clean:<15} {coef:>12.4f} {se:>12.4f} {pval:>10.4f} {stars:>12}")
    
    # Print each group
    print_coef_group("EFFECTS OF INITIAL UNEMPLOYMENT RATE (BACHELOR'S)", ur_coefs)
    print_coef_group("EFFECTS OF MASTER'S DEGREE", master_coefs)
    print_coef_group("DIFFERENTIAL EFFECTS OF INITIAL UNEMPLOYMENT RATE FOR MASTER'S", master_ur_coefs)
    
    # Print model statistics
    print("\n" + "="*80)
    print("MODEL STATISTICS")
    print("="*80)
    print(f"R-squared: {results.rsquared:.4f}")
    print(f"Adjusted R-squared: {results.rsquared_adj:.4f}")
    print(f"Number of observations: {results.nobs}")
    print(f"F-statistic: {results.fvalue:.2f}")
    print(f"Prob(F-statistic): {results.f_pvalue:.4f}")
    
    return

# Helper function to create evenly spaced bins
def create_even_bins(exp_range=(3,14), bin_size=3):
    """Generate evenly spaced bins of specified size"""
    bins = []
    for start in range(exp_range[0], exp_range[1] + 1, bin_size):
        end = min(start + bin_size - 1, exp_range[1])
        bins.append((start, end))
    return bins

# # Usage example with even bins
# exp_groups = create_even_bins(exp_range=(3,14), bin_size=3)
# # Should generate: [(3,5), (6,8), (9,11), (12,14)]

# Usage example with custom bins
custom_exp_groups = [(3,4),(5,6),(7,8),(9,10),(11,12),(13,14)]  # Custom uneven grouping
custom_exp_groups = [(3,5),(6,8),(9,11),(12,14)]  # Custom uneven grouping

# custom_exp_groups = [(3,6),(7,11),(12,14)]  # Custom uneven grouping

# Prepare data with custom groups
df_processed_oreo_custom = prepare_data_oreo_flexible_groups(df_clean_salary, custom_exp_groups)

# Create regression formula
ols_formula_oreo_custom = create_oreo_formula_flexible_groups(custom_exp_groups)

# Run regression
ols_results_oreo_custom = run_ols_robust_analysis(df_processed_oreo_custom, ols_formula_oreo_custom)

print(ols_results_oreo_custom.summary())
# Display results
display_grouped_results_flexible(ols_results_oreo_custom, custom_exp_groups)

# Plot results
fig = plot_marginal_effects_flexible_groups(ols_results_oreo_custom, custom_exp_groups)